### PSD Pipeline
#### Table of Contents:
* [1. Environment Setup](#1-environment-setup)
    * [1.1 Library Imports](#11-library-imports)
* [2. Data Segmentation](#data-segmentation)
* [3. Feature Extraction and Class Distribution](#3-feature-extraction--class-distribution)
    * [3.1 PSD](#31-psd-feature-extraction)
    * [3.2 Class Distribution](#32-class-distribution)
* [4. Classification Schemes](#4-classification-schemes)
    * [4.1 Emotional vs. Neutral](#41-emotional-vs-neutral-remap--class-distribution)
    * [4.2 Positive vs. Negative](#42-positive-vs-negative-remap--class-distribution)



### 1. Environment Setup

##### 1.1 Library Imports

In [ ]:
import numpy as np
import pandas as pd
from scipy.io import loadmat
from pathlib import Path

from scipy.signal import welch
import optuna

from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler


##### 1.2 Dataset loading/inspection

In [ ]:
dataset_path = Path('DEED')
eeg_dataset = []
for file in dataset_path.iterdir():
    mat = loadmat(file)
    eeg = mat['Data']
    fname = file.stem  
    label_part = [part for part in fname.split("_") if part.startswith("E")][0]
    label = int(label_part[1:])  
    eeg_dataset.append((eeg, label))

print(f"Loaded {len(eeg_dataset)} trials.")
print("Example shapes:", [(arr.shape, lbl) for arr, lbl in eeg_dataset[:3]])

print("\nSample filename → label mapping:")
for file, (_, label) in zip(dataset_path.iterdir(), eeg_dataset[:10]):
    print(f"  {file.stem} → E{label}")

Loaded 533 trials.
Example shapes: [((6, 290000), 2), ((6, 36000), 2), ((6, 51000), 3)]


### 2. Data Segmentation
20s windows

In [22]:
def segmentation(eeg_dataset, window_sec, fs):
    window_size = int(window_sec * fs)
    X = []
    y = []

    for eeg_array, label in eeg_dataset:
        n_samples = eeg_array.shape[1]
        start = 0
        while start + window_size <= n_samples:
            window = eeg_array[:, start:start + window_size]
            X.append(window)
            y.append(label)
            start += window_size  

    return X, y

# secseg2, secseg2_labels = segmentation(eeg_dataset, 2, 200)
# secseg10, secseg10_labels = segmentation(eeg_dataset, 10, 200)
secseg20, secseg20_labels = segmentation(eeg_dataset, 20, 200)
print(f"=== Total 20 second windows === \n {len(secseg20)}")

=== Total 20 second windows === 
 7490


### 3. Feature Extraction & Class Distribution

##### 3.1 PSD Feature Extraction

In [6]:
def extract_psd_features(segmented_windows, labels, fs=200):
    freq_bands = {'delta': (0.5, 4),
                'theta': (4, 8),
                'alpha': (8, 12),
                'beta': (12, 30),
                'gamma': (30, 45)
                }
    
    windows = np.array(segmented_windows)
    n_windows, n_channels, n_samples = windows.shape
    
    nperseg = min(512, n_samples)  
    noverlap = nperseg // 2
    
    all_features = []
    
    for ch_idx in range(n_channels):
        ch_data = windows[:, ch_idx, :]
        
        f, Pxx = welch(ch_data, fs=fs, nperseg=nperseg, noverlap=noverlap, axis=1)
        
        ch_features = []
        for band_name, (low, high) in freq_bands.items():
            idx = np.logical_and(f >= low, f <= high)
            band_power = np.mean(Pxx[:, idx], axis=1)  
            ch_features.append(band_power)
        
        all_features.append(np.column_stack(ch_features))
    
    X = np.hstack(all_features)
    y = np.array(labels)
    
    return X, y

##### 3.2 Class Distribution

In [ ]:
X20, y20 = extract_psd_features(secseg20, secseg20_labels)
print(X20.shape)
print(y20.shape) 

print("\n=== Total Class Distribution===")
unique, counts = np.unique(y20, return_counts=True)
for label, count in zip(unique, counts):
    print(f"E{label}: {count} windows ({count/len(y20)*100:.1f}%)")

(7490, 30)
(7490,)

=== Class Distribution (20s window) ===
E0: 1294 windows (17.3%)
E1: 305 windows (4.1%)
E2: 1069 windows (14.3%)
E3: 2855 windows (38.1%)
E4: 1698 windows (22.7%)
E5: 269 windows (3.6%)


### 4. Classification Schemes

##### 4.1 Emotional vs. Neutral (remap + class distribution)  
Emotional {E1, E2, E4, E5} vs. Neutral Dream {E3} 

In [8]:
def remap_emotional_neutral(y):
    y = np.array(y)
    keep_mask = np.isin(y, [1, 2, 3, 4, 5])  
    new_y = np.zeros(len(y), dtype=int)
    new_y[y == 3] = 0  
    new_y[np.isin(y, [1, 2, 4, 5])] = 1  
    return new_y[keep_mask], keep_mask

In [ ]:
y20_emotional_neutral, mask20 = remap_emotional_neutral(y20)
X20_emotional_neutral = X20[mask20]
print(f"\n=== Emotional vs. Negative Class Distribution ===")
print(f"  Total: {len(y20_emotional_neutral)}")
print(f"  Neutral (0): {np.sum(y20_emotional_neutral == 0)} ({np.sum(y20_emotional_neutral == 0)/len(y20_emotional_neutral)*100:.1f}%)")
print(f"  Emotional (1): {np.sum(y20_emotional_neutral == 1)} ({np.sum(y20_emotional_neutral == 1)/len(y20_emotional_neutral)*100:.1f}%)")


20s windows:
  Total: 6196
  Neutral (0): 2855 (46.1%)
  Emotional (1): 3341 (53.9%)


##### 4.2 Positive vs. Negative (remap + class distribution)
Positive {E4, E5} vs. Negative {E1, E2}

In [10]:
def remap_positive_negative(y):
    y = np.array(y)
    keep_mask = np.isin(y, [1, 2, 4, 5])  
    new_y = np.zeros(len(y), dtype=int)
    new_y[np.isin(y, [1, 2])] = 0  
    new_y[np.isin(y, [4, 5])] = 1  
    return new_y[keep_mask], keep_mask

In [18]:
y20_positive_negative, mask20 = remap_positive_negative(y20)
X20_positive_negative = X20[mask20]
print(f"\n=== Positive vs. Negative Class Distribution ===")
print(f"  Total: {len(y20_positive_negative)}")
print(f"  Negative (0): {np.sum(y20_positive_negative == 0)} ({np.sum(y20_positive_negative == 0)/len(y20_positive_negative)*100:.1f}%)")
print(f"  Positive (1): {np.sum(y20_positive_negative == 1)} ({np.sum(y20_positive_negative == 1)/len(y20_positive_negative)*100:.1f}%)")


=== Positive vs. Negative Class Distribution ===
  Total: 3341
  Negative (0): 1374 (41.1%)
  Positive (1): 1967 (58.9%)


## XGBoost
- 80/20 train test split

In [14]:
# Emotional vs neutral
def xgboost_training_loop():
    X_train, X_test, y_train, y_test = train_test_split(
        X20_emotional_neutral, y20_emotional_neutral, 
        test_size=0.2, 
        random_state=42, 
        stratify=y20_emotional_neutral 
    )
    bst = XGBClassifier()  
    bst.fit(X_train, y_train)
    preds = bst.predict(X_test)

    print("Emotional vs Neutral Baseline Model")
    print(f"Accuracy: {accuracy_score(y_test, preds):.4f}")
    print(f"\nConfusion Matrix:")
    print(confusion_matrix(y_test, preds))
    print(f"\nClassification Report:")
    print(classification_report(y_test, preds, target_names=['Neutral', 'Emotional']))

xgboost_training_loop() 

Emotional vs Neutral Baseline Model
Accuracy: 0.6815

Confusion Matrix:
[[374 197]
 [198 471]]

Classification Report:
              precision    recall  f1-score   support

     Neutral       0.65      0.65      0.65       571
   Emotional       0.71      0.70      0.70       669

    accuracy                           0.68      1240
   macro avg       0.68      0.68      0.68      1240
weighted avg       0.68      0.68      0.68      1240



In [15]:
# Positive vs Negative
def xgboost_training_loop_20s():
    X_train, X_test, y_train, y_test = train_test_split(
        X20_positive_negative, y20_positive_negative, 
        test_size=0.2, 
        random_state=42, 
        stratify=y20_positive_negative
    )
    bst = XGBClassifier()  
    bst.fit(X_train, y_train)
    preds = bst.predict(X_test)

    print("Positive vs. Negative Baseline Model")
    print(f"Accuracy: {accuracy_score(y_test, preds):.4f}")
    print(f"\nConfusion Matrix:")
    print(confusion_matrix(y_test, preds))
    print(f"\nClassification Report:")
    print(classification_report(y_test, preds, target_names=['Neutral', 'Emotional']))

xgboost_training_loop_20s() 

Positive vs. Negative Baseline Model
Accuracy: 0.6607

Confusion Matrix:
[[132 143]
 [ 84 310]]

Classification Report:
              precision    recall  f1-score   support

     Neutral       0.61      0.48      0.54       275
   Emotional       0.68      0.79      0.73       394

    accuracy                           0.66       669
   macro avg       0.65      0.63      0.63       669
weighted avg       0.65      0.66      0.65       669

